In [ ]:
"""
TRAINING NOTEBOOK FOR IMBALANCED SUNT DATASET
==============================================
"""

import pandas as pd
import numpy as np
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')

print("✅ Libraries imported")


Mounted at /content/drive
✅ Libraries imported


In [ ]:
# ============================================
# CELL 1: LOAD DATA
# ============================================

DRIVE_PATH = '/content/drive/MyDrive/Occupancy_capstone/Dataset/'
X_PATH = DRIVE_PATH + 'sunt_X_2024_03_march.parquet'
Y_PATH = DRIVE_PATH + 'sunt_y_2024_03_march.pkl'
# Option A: If you have X and y saved
X = pd.read_parquet(X_PATH)
with open(Y_PATH, 'rb') as f:
    y = pickle.load(f)

print("\n" + "=" * 70)
print("📊 DATASET SUMMARY")
print("=" * 70)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts(normalize=True).sort_index())



📊 DATASET SUMMARY
X shape: (8544587, 14)
y shape: (8544587,)

Target distribution:
occupancy_level
high         0.116677
low          0.577081
medium       0.272316
very_high    0.033927
Name: proportion, dtype: float64


In [ ]:
# ============================================
# CELL 2: ANALYZE CLASS IMBALANCE
# ============================================

print("\n" + "=" * 70)
print("⚠️  CLASS IMBALANCE ANALYSIS")
print("=" * 70)

# Calculate imbalance ratio
class_counts = y.value_counts()
majority_class = class_counts.max()
minority_class = class_counts.min()
imbalance_ratio = majority_class / minority_class

print(f"\nClass counts:")
for cls, count in class_counts.sort_index().items():
    pct = count / len(y) * 100
    bar = "█" * int(pct / 2)
    print(f"   {cls:10s}: {count:>10,} ({pct:5.2f}%) {bar}")

print(f"\nImbalance ratio: {imbalance_ratio:.1f}:1")
print(f"Majority class: {class_counts.idxmax()} ({class_counts.max():,})")
print(f"Minority class: {class_counts.idxmin()} ({class_counts.min():,})")

if imbalance_ratio > 10:
    print("\n⚠️  SEVERE IMBALANCE DETECTED!")
    print("   Using class_weight='balanced' is CRITICAL")
elif imbalance_ratio > 5:
    print("\n⚠️  Moderate imbalance detected")
    print("   Recommend using class_weight='balanced'")
else:
    print("\n✅ Acceptable class balance")


⚠️  CLASS IMBALANCE ANALYSIS

Class counts:
   high      :    996,957 (11.67%) █████
   low       :  4,930,915 (57.71%) ████████████████████████████
   medium    :  2,326,825 (27.23%) █████████████
   very_high :    289,890 ( 3.39%) █

Imbalance ratio: 17.0:1
Majority class: low (4,930,915)
Minority class: very_high (289,890)

⚠️  SEVERE IMBALANCE DETECTED!
   Using class_weight='balanced' is CRITICAL


In [ ]:
# ============================================
# OPCIÓN A: NO BALANCEAR + CLASS WEIGHTS (RECOMENDADO)
# ============================================

print("\n" + "=" * 70)
print("⚖️  CLASS BALANCE STRATEGY: WEIGHTED TRAINING")
print("=" * 70)

print("\n📊 Current class distribution:")
print("-" * 70)
dist = y.value_counts(normalize=True).sort_index()
for cls, prop in dist.items():
    count = (y == cls).sum()
    bar = "█" * int(prop * 50)
    print(f"   {cls:>10s}: {count:>10,} ({prop*100:>5.1f}%) {bar}")

# Calculate class weights
classes = np.unique(y)
class_weights_array = compute_class_weight(
    'balanced',
    classes=classes,
    y=y
)
class_weights = dict(zip(classes, class_weights_array))

print("\n📊 Computed class weights:")
print("-" * 70)
for cls, weight in sorted(class_weights.items()):
    print(f"   {cls:>10s}: {weight:>6.3f}")

print("\n💡 Interpretation:")
print("   • Higher weight = Model will pay more attention to this class")
print("   • Low weight = Majority class (less important)")
print("   • This balances without throwing away data!")

print(f"\n✅ Strategy: Use ALL {len(X):,} records with class weights")
print("   (No data is discarded)")
print("=" * 70)

# ============================================
# VERIFICAR DISTRIBUCIÓN FINAL
# ============================================

print("\n" + "=" * 70)
print("📊 FINAL DATASET FOR TRAINING")
print("=" * 70)
print(f"   Total records: {len(X):,}")
print(f"   Total features: {len(X.columns)}")
print(f"   Strategy: {'Weighted (all data)' if len(X) > 1000000 else 'Undersampled'}")
print("=" * 70)


⚖️  CLASS BALANCE STRATEGY: WEIGHTED TRAINING

📊 Current class distribution:
----------------------------------------------------------------------
         high:    996,957 ( 11.7%) █████
          low:  4,930,915 ( 57.7%) ████████████████████████████
       medium:  2,326,825 ( 27.2%) █████████████
    very_high:    289,890 (  3.4%) █

📊 Computed class weights:
----------------------------------------------------------------------
         high:  2.143
          low:  0.433
       medium:  0.918
    very_high:  7.369

💡 Interpretation:
   • Higher weight = Model will pay more attention to this class
   • Low weight = Majority class (less important)
   • This balances without throwing away data!

✅ Strategy: Use ALL 8,544,587 records with class weights
   (No data is discarded)

📊 FINAL DATASET FOR TRAINING
   Total records: 8,544,587
   Total features: 14
   Strategy: Weighted (all data)


In [ ]:
# ============================================
# CELL 3: TRAIN/TEST SPLIT (STRATIFIED)
# ============================================

print("\n" + "=" * 70)
print("✂️  TRAIN/TEST SPLIT (STRATIFIED)")
print("=" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # ⭐ IMPORTANT: Maintains class distribution
)

print(f"\nTrain set: {len(X_train):,} records")
print(f"Test set:  {len(X_test):,} records")

print(f"\nTrain distribution:")
for cls, pct in y_train.value_counts(normalize=True).sort_index().items():
    print(f"   {cls}: {pct*100:.2f}%")

print(f"\nTest distribution:")
for cls, pct in y_test.value_counts(normalize=True).sort_index().items():
    print(f"   {cls}: {pct*100:.2f}%")


✂️  TRAIN/TEST SPLIT (STRATIFIED)

Train set: 6,835,669 records
Test set:  1,708,918 records

Train distribution:
   high: 11.67%
   low: 57.71%
   medium: 27.23%
   very_high: 3.39%

Test distribution:
   high: 11.67%
   low: 57.71%
   medium: 27.23%
   very_high: 3.39%


In [ ]:
# ============================================
# CELL 4: TRAIN MODEL WITH CLASS WEIGHTS
# ============================================

import time # Import the time module

print("\n" + "=" * 70)
print("🤖 TRAINING RANDOM FOREST WITH CLASS WEIGHTS")
print("=" * 70)

# Usar los class_weights calculados anteriormente
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight=class_weights,  # ✅ Usar weights calculados (más preciso)
    # class_weight='balanced',    # ⚠️ También funciona, pero menos control
    random_state=42,
    n_jobs=-1,
    verbose=1  # Cambié a 1 para ver progreso
)

print(f"\nModel: {model.__class__.__name__}")
print(f"n_estimators: {model.n_estimators}")
print(f"max_depth: {model.max_depth}")
print(f"class_weight: custom weights ✅")
print(f"\nClass weights being used:")
for cls, weight in sorted(class_weights.items()):
    print(f"   {cls:>10s}: {weight:>6.3f}")

print("\n⏳ Training... (this may take 5-10 minutes)")
start_time = time.time()

model.fit(X_train, y_train)

elapsed = time.time() - start_time
print(f"\n✅ Training completed in {elapsed:.2f} seconds ({elapsed/60:.1f} minutes)")


🤖 TRAINING RANDOM FOREST WITH CLASS WEIGHTS

Model: RandomForestClassifier
n_estimators: 100
max_depth: 15
class_weight: custom weights ✅

Class weights being used:
         high:  2.143
          low:  0.433
       medium:  0.918
    very_high:  7.369

⏳ Training... (this may take 5-10 minutes)


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=-1)]: Done  46 tasks      | elapsed:  7.5min



✅ Training completed in 996.16 seconds (16.6 minutes)


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed: 16.3min finished


In [ ]:
# ============================================
# CELL 5: EVALUATION WITH PROPER METRICS
# ============================================

print("\n" + "=" * 70)
print("📊 MODEL EVALUATION")
print("=" * 70)

# Predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Standard accuracy
acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

# ⭐ BALANCED ACCURACY - Better metric for imbalanced data
bal_acc_train = balanced_accuracy_score(y_train, y_pred_train)
bal_acc_test = balanced_accuracy_score(y_test, y_pred_test)

# F1 Score (weighted)
f1_train = f1_score(y_train, y_pred_train, average='weighted')
f1_test = f1_score(y_test, y_pred_test, average='weighted')

# F1 Score (macro - treats all classes equally)
f1_macro_train = f1_score(y_train, y_pred_train, average='macro')
f1_macro_test = f1_score(y_test, y_pred_test, average='macro')

print("\n" + "-" * 70)
print("METRICS COMPARISON:")
print("-" * 70)
print(f"{'Metric':<25} {'Train':>12} {'Test':>12}")
print("-" * 70)
print(f"{'Accuracy':<25} {acc_train:>12.4f} {acc_test:>12.4f}")
print(f"{'Balanced Accuracy ⭐':<25} {bal_acc_train:>12.4f} {bal_acc_test:>12.4f}")
print(f"{'F1 (weighted)':<25} {f1_train:>12.4f} {f1_test:>12.4f}")
print(f"{'F1 (macro) ⭐':<25} {f1_macro_train:>12.4f} {f1_macro_test:>12.4f}")
print("-" * 70)

print("\n💡 KEY METRICS FOR IMBALANCED DATA:")
print(f"   • Balanced Accuracy: {bal_acc_test:.4f}")
print(f"   • F1 Macro: {f1_macro_test:.4f}")
print("   (These are more meaningful than regular accuracy)")


📊 MODEL EVALUATION


[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:   50.7s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:  1.7min finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  46 tasks      | elapsed:   12.0s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:   25.3s finished



----------------------------------------------------------------------
METRICS COMPARISON:
----------------------------------------------------------------------
Metric                           Train         Test
----------------------------------------------------------------------
Accuracy                        0.5982       0.5920
Balanced Accuracy ⭐             0.5783       0.5677
F1 (weighted)                   0.6165       0.6103
F1 (macro) ⭐                    0.4834       0.4753
----------------------------------------------------------------------

💡 KEY METRICS FOR IMBALANCED DATA:
   • Balanced Accuracy: 0.5677
   • F1 Macro: 0.4753
   (These are more meaningful than regular accuracy)


In [ ]:
# ============================================
# CELL 6: DETAILED CLASSIFICATION REPORT
# ============================================

print("\n" + "=" * 70)
print("📋 CLASSIFICATION REPORT (TEST SET)")
print("=" * 70)
print(classification_report(y_test, y_pred_test))


📋 CLASSIFICATION REPORT (TEST SET)
              precision    recall  f1-score   support

        high       0.31      0.36      0.33    199392
         low       0.83      0.70      0.76    986183
      medium       0.44      0.44      0.44    465365
   very_high       0.24      0.77      0.37     57978

    accuracy                           0.59   1708918
   macro avg       0.46      0.57      0.48   1708918
weighted avg       0.65      0.59      0.61   1708918



In [ ]:
# ============================================
# CELL 7: CONFUSION MATRIX
# ============================================

print("\n" + "=" * 70)
print("📊 CONFUSION MATRIX (TEST SET)")
print("=" * 70)

cm = confusion_matrix(y_test, y_pred_test)
labels = sorted(y.unique())

print(f"\n{'':>12}", end="")
for label in labels:
    print(f"{label:>12}", end="")
print(" ← Predicted")
print("-" * (12 + 12 * len(labels)))

for i, label in enumerate(labels):
    print(f"{label:>12}", end="")
    for j in range(len(labels)):
        print(f"{cm[i,j]:>12,}", end="")
    print(f"  | {label}")

print("\n↑ Actual")

# Calculate per-class accuracy
print("\n📊 Per-class accuracy:")
for i, label in enumerate(labels):
    class_total = cm[i, :].sum()
    class_correct = cm[i, i]
    class_acc = class_correct / class_total if class_total > 0 else 0
    print(f"   {label:>10}: {class_acc:.2%} ({class_correct:,}/{class_total:,})")



📊 CONFUSION MATRIX (TEST SET)

                    high         low      medium   very_high ← Predicted
------------------------------------------------------------
        high      71,612      16,801      48,045      62,934  | high
         low      61,010     691,698     204,315      29,160  | low
      medium      92,221     121,448     203,627      48,069  | medium
   very_high       8,690       1,130       3,355      44,803  | very_high

↑ Actual

📊 Per-class accuracy:
         high: 35.92% (71,612/199,392)
          low: 70.14% (691,698/986,183)
       medium: 43.76% (203,627/465,365)
    very_high: 77.28% (44,803/57,978)


In [ ]:
# ============================================
# CELL 8: FEATURE IMPORTANCE
# ============================================

print("\n" + "=" * 70)
print("🔝 FEATURE IMPORTANCE")
print("=" * 70)

importances = model.feature_importances_
feature_names = X.columns.tolist()
indices = np.argsort(importances)[::-1]

print(f"\nTop 15 most important features:")
print("-" * 70)
for i in range(min(15, len(feature_names))):
    idx = indices[i]
    bar = "█" * int(importances[idx] * 50)
    print(f"{i+1:2d}. {feature_names[idx]:<25} {importances[idx]:.4f} {bar}")

# Cumulative importance
print(f"\n📊 Cumulative importance:")
cumsum = 0
for i in range(len(importances)):
    cumsum += importances[indices[i]]
    if cumsum >= 0.5 and i < 10:
        print(f"   Top {i+1} features explain 50% of importance")
    if cumsum >= 0.8:
        print(f"   Top {i+1} features explain 80% of importance")
        break



🔝 FEATURE IMPORTANCE

Top 15 most important features:
----------------------------------------------------------------------
 1. hour                      0.2192 ██████████
 2. trip_number               0.1303 ██████
 3. route_short_name          0.1169 █████
 4. register_code             0.0936 ████
 5. pt_sequence               0.0868 ████
 6. route_percentage          0.0591 ██
 7. is_rush_hour              0.0589 ██
 8. direction_id              0.0574 ██
 9. time_of_day               0.0552 ██
10. end_trip                  0.0512 ██
11. day_of_week               0.0284 █
12. is_weekend                0.0226 █
13. trip_stage                0.0204 █
14. month                     0.0000 

📊 Cumulative importance:
   Top 4 features explain 50% of importance
   Top 5 features explain 50% of importance
   Top 6 features explain 50% of importance
   Top 7 features explain 50% of importance
   Top 8 features explain 50% of importance
   Top 8 features explain 80% of importance


In [ ]:
# ============================================
# CELL 9: COMPARE WITH BASELINE
# ============================================

print("\n" + "=" * 70)
print("📈 COMPARISON WITH BASELINE")
print("=" * 70)

# Baseline: always predict majority class
majority_class = y_train.value_counts().idxmax()
baseline_acc = (y_test == majority_class).mean()

print(f"\nBaseline (always predict '{majority_class}'):")
print(f"   Accuracy: {baseline_acc:.4f}")

print(f"\nYour model:")
print(f"   Accuracy: {acc_test:.4f} ({(acc_test - baseline_acc)*100:+.2f}% vs baseline)")
print(f"   Balanced Accuracy: {bal_acc_test:.4f}")

improvement = (acc_test - baseline_acc) / baseline_acc * 100
print(f"\n{'✅' if improvement > 0 else '❌'} Model improvement: {improvement:+.2f}%")

if bal_acc_test > 0.5:
    print("✅ Balanced accuracy > 50% (better than random)")
else:
    print("⚠️  Balanced accuracy < 50% (model needs improvement)")


📈 COMPARISON WITH BASELINE

Baseline (always predict 'low'):
   Accuracy: 0.5771

Your model:
   Accuracy: 0.5920 (+1.50% vs baseline)
   Balanced Accuracy: 0.5677

✅ Model improvement: +2.59%
✅ Balanced accuracy > 50% (better than random)


In [ ]:
# ============================================
# CELL 10: FINAL SUMMARY
# ============================================

print("\n" + "=" * 70)
print("🎉 FINAL SUMMARY")
print("=" * 70)

print(f"""
📊 DATASET:
   • Total records: {len(X):,}
   • Features: {len(X.columns)}
   • Classes: {len(labels)} ({', '.join(labels)})
   • Imbalance ratio: {imbalance_ratio:.1f}:1

🤖 MODEL:
   • Algorithm: Random Forest
   • class_weight: balanced
   • n_estimators: {model.n_estimators}

📈 PERFORMANCE:
   • Accuracy: {acc_test:.4f}
   • Balanced Accuracy: {bal_acc_test:.4f} ⭐
   • F1 Macro: {f1_macro_test:.4f} ⭐
   • Baseline: {baseline_acc:.4f}
   • Improvement: {improvement:+.2f}%

💡 INTERPRETATION:
   • Balanced Accuracy of {bal_acc_test:.2%} means the model
     correctly classifies ~{bal_acc_test*100:.0f}% across ALL classes
   • This is {'GOOD' if bal_acc_test > 0.6 else 'ACCEPTABLE' if bal_acc_test > 0.5 else 'NEEDS IMPROVEMENT'} for imbalanced data
""")

print("=" * 70)
print("✅ Training complete!")
print("=" * 70)

# ============================================
# OPTIONAL: SAVE MODEL
# ============================================


# to save the trained model
import pickle

model_filename = 'sunt_rf_model_balanced.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(model, f)
print(f"✅ Model saved to {model_filename}")



🎉 FINAL SUMMARY

📊 DATASET:
   • Total records: 8,544,587
   • Features: 14
   • Classes: 4 (high, low, medium, very_high)
   • Imbalance ratio: 17.0:1

🤖 MODEL:
   • Algorithm: Random Forest
   • class_weight: balanced
   • n_estimators: 100

📈 PERFORMANCE:
   • Accuracy: 0.5920
   • Balanced Accuracy: 0.5677 ⭐
   • F1 Macro: 0.4753 ⭐
   • Baseline: 0.5771
   • Improvement: +2.59%

💡 INTERPRETATION:
   • Balanced Accuracy of 56.77% means the model
     correctly classifies ~57% across ALL classes
   • This is ACCEPTABLE for imbalanced data

✅ Training complete!
✅ Model saved to sunt_rf_model_balanced.pkl


In [ ]:
feature_names